# Corpus Expansion: Mistral-7B Probe

Cross-probe replication of the Llama corpus_expansion run on the same corpora.
Outputs cache files into a sibling Mistral cache dir so reconcile_metrics_corpus_expansion.py
can be pointed at either probe to summarize that probe's results without mixing.

**Output dir:** `My Drive/LRTIA/Results/corpus_expansion/mistral/<corpus_id>.json`

In [ ]:
!pip install -q -U bitsandbytes>=0.46.1 accelerate

import numpy as np
import json, math, os, gc, random, time
from pathlib import Path
from collections import Counter
from scipy import stats
from scipy.ndimage import uniform_filter1d
from tqdm.auto import tqdm
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

from google.colab import drive
drive.mount('/content/drive')

# Mistral cache lives in a sibling dir to llama/ — keeps results separated by probe.
BASE = Path('/content/drive/MyDrive/LRTIA/Results/corpus_expansion/mistral')
BASE.mkdir(parents=True, exist_ok=True)
CORPUS_DIR = Path('/content/drive/MyDrive/LRTIA/Data/corpus_expansion/clean')
TARGETS_PATH = Path('/content/drive/MyDrive/LRTIA/Results/corpus_expansion/targets_llama.jsonl')

# Mistral-7B base. Token boundaries differ from Llama — the existing
# targets_llama.jsonl positions are computed on Llama-tokenized text, so
# they're approximate for Mistral. Recommend using the dynamic positioning
# path (pick targets at fractions of the doc) instead of trusting the
# pre-computed start/end token offsets in targets_llama. The current run loop
# clamps and skips on overrun, so this just means some target positions will
# shift slightly from the Llama run — same documents, very-slightly-different
# 100-token windows around the same fractional positions.
MODEL_NAME = 'mistralai/Mistral-7B-v0.1'
C = 100
N_SHUFFLES = 1
SEED = 20260429

# === Run order (mirrors Llama notebook) ===
# Phase 1: English genres (4 corpora)
# Phase 2: Multilingual TED transcripts (6 corpora)
# Phase 3: FLORES (skip — bag-of-sentences chunking, short-range only)
PHASE = 1  # Change to 2 for multilingual TED

if PHASE == 1:
    RUN_CORPORA = ['gutenberg_fiction_en', 'news_en', 'subtitles_dialogue_en', 'ted_transcripts_en']
elif PHASE == 2:
    RUN_CORPORA = [
        'ted_transcripts_ar', 'ted_transcripts_de', 'ted_transcripts_es',
        'ted_transcripts_fr', 'ted_transcripts_ru', 'ted_transcripts_tr',
    ]
elif PHASE == 3:
    RUN_CORPORA = [f'flores_matched_{lang}' for lang in
        ['en','ar','de','es','fi','fr','he','hi','ja','ko','ru','sw','tr','vi','zh']]

print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'Probe: {MODEL_NAME}')
print(f'Phase {PHASE}: {len(RUN_CORPORA)} corpora')
print(f'Corpora: {RUN_CORPORA}')
print('Setup done')

In [ ]:
# === Load targets manifest ===
# If targets file is on Drive, use it. Otherwise upload.
if TARGETS_PATH.exists():
    all_targets = []
    with open(TARGETS_PATH) as f:
        for line in f:
            all_targets.append(json.loads(line))
    print(f'Loaded {len(all_targets)} targets')
else:
    print(f'Targets file not found at {TARGETS_PATH}')
    print('Upload targets_llama.jsonl to Drive at:')
    print('  LRTIA/Results/corpus_expansion/targets_llama.jsonl')
    raise FileNotFoundError('Upload targets first')

# Group by corpus
targets_by_corpus = {}
for t in all_targets:
    targets_by_corpus.setdefault(t['corpus_id'], []).append(t)

for c in RUN_CORPORA:
    n = len(targets_by_corpus.get(c, []))
    print(f'  {c}: {n} targets')

In [ ]:
# === Load model ===
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.float16),
    device_map='auto'
)
model.eval()
print(f'{MODEL_NAME} loaded')

In [ ]:
# === Pipeline functions ===

@torch.no_grad()
def ppl_nll(ctx_toks, tgt_toks):
    if len(tgt_toks) < 2: return float('inf'), float('inf')
    full = list(ctx_toks) + list(tgt_toks)
    ts = len(ctx_toks)
    ids = torch.tensor([full], device=model.device)
    out = model(ids); logits = out.logits[0]
    nll = 0.0; cnt = 0
    for i in range(ts, len(full)-1):
        lp = torch.log_softmax(logits[i], dim=-1)
        nll += -lp[full[i+1]].item(); cnt += 1
    del out, logits; torch.cuda.empty_cache()
    if cnt == 0: return float('inf'), float('inf')
    mn = nll/cnt; return math.exp(mn), mn

def compute_corrected_curves(ctx, tgt):
    mc = len(ctx)
    o_ppl, o_nll, s_ppl, s_nll = [], [], [], []
    for c in range(mc+1):
        pfx = ctx[-c:] if c > 0 else []
        p, n = ppl_nll(pfx, tgt)
        o_ppl.append(p); o_nll.append(n)
        if c == 0:
            s_ppl.append(p); s_nll.append(n)
        else:
            rng = random.Random(SEED + c)
            sp_l, sn_l = [], []
            for _ in range(N_SHUFFLES):
                sh = list(pfx); rng.shuffle(sh)
                sp, sn = ppl_nll(sh, tgt)
                if not math.isinf(sp): sp_l.append(sp); sn_l.append(sn)
            s_ppl.append(np.mean(sp_l) if sp_l else p)
            s_nll.append(np.mean(sn_l) if sn_l else n)
    dists = list(range(1, mc+1))
    mo = [o_ppl[d-1]-o_ppl[d] for d in dists]
    ms = [s_ppl[d-1]-s_ppl[d] for d in dists]
    delta = [a-b for a,b in zip(mo,ms)]
    mo_n = [o_nll[d-1]-o_nll[d] for d in dists]
    ms_n = [s_nll[d-1]-s_nll[d] for d in dists]
    delta_n = [a-b for a,b in zip(mo_n,ms_n)]
    return {
        'distances': dists,
        'ordered_ppl': o_ppl, 'shuffled_ppl': s_ppl,
        'delta_ppl': delta,
        'ordered_nll': o_nll, 'shuffled_nll': s_nll,
        'delta_nll': delta_n,
    }

# Power law fit
BIN_EDGES = [1, 2, 3, 4, 5, 7, 10, 15, 20, 30, 50, 75, 100]
def fit_pl(marg):
    bm, bc = [], []
    for i in range(len(BIN_EDGES)-1):
        lo, hi = BIN_EDGES[i], BIN_EDGES[i+1]
        vals = marg[lo-1:hi-1]; vals = vals[~np.isnan(vals)]
        if len(vals) > 0 and np.mean(vals) > 0:
            bm.append(np.mean(vals)); bc.append((lo+hi)/2)
    if len(bm) >= 4:
        s,i,r,p,_ = stats.linregress(np.log(bc), np.log(bm))
        return s, r
    return None, None

# Path fixer: local manifest paths -> Drive paths
# Local manifest has: data/corpus_expansion/clean/X/file.txt
# Drive has:          /content/drive/MyDrive/LRTIA/Data/corpus_expansion/X/file.txt
# Strip 'clean/' from the path
def fix_path(p):
    return str(p).replace('data/corpus_expansion/clean/',
        '/content/drive/MyDrive/LRTIA/Data/corpus_expansion/')

# Verify fix_path works
_test = fix_path('data/corpus_expansion/clean/gutenberg_fiction_en/test.txt')
assert 'clean' not in _test, f'fix_path still has clean: {_test}'
print(f'Path fix verified: clean/ stripped correctly')
print(f'Functions ready — {N_SHUFFLES} shuffle')

In [ ]:
# === Main run loop ===
# IMPORTANT: targets_llama.jsonl positions are Llama-token offsets, NOT Mistral-token
# offsets. We re-tokenize each doc with Mistral and reuse only the document_id,
# target_id, target_position_fraction, language, genre, modality. Target positions
# are recomputed in Mistral-token coordinates from the fraction, so this run
# tests the same documents at the same fractional positions but in the Mistral
# tokenizer's coordinate system.

TARGET_LEN = 30  # tokens

def fix_path(p):
    return str(p).replace('data/corpus_expansion/clean/',
        '/content/drive/MyDrive/LRTIA/Data/corpus_expansion/')

doc_token_cache = {}

def get_doc_tokens(file_path):
    if file_path not in doc_token_cache:
        fp = fix_path(file_path)
        if not Path(fp).exists():
            fp = str(Path('/content/drive/MyDrive/LRTIA/Data/corpus_expansion') /
                     '/'.join(file_path.split('/')[-2:]))
        text = Path(fp).read_text(encoding='utf-8', errors='replace').strip()
        doc_token_cache[file_path] = tokenizer.encode(text, add_special_tokens=False)
    return doc_token_cache[file_path]

for corpus_id in RUN_CORPORA:
    cache_path = BASE / f'{corpus_id}.json'
    if cache_path.exists():
        with open(cache_path) as f: n = len(json.load(f))
        print(f'\n{corpus_id}: cached ({n})'); continue

    targets = targets_by_corpus.get(corpus_id, [])
    if not targets:
        print(f'\n{corpus_id}: no targets'); continue

    print(f'\n{"="*60}')
    print(f'{corpus_id} ({len(targets)} targets)')
    print(f'{"="*60}')

    # Group targets by document so we tokenize each doc once.
    by_doc = {}
    for t in targets:
        by_doc.setdefault(t['document_id'], []).append(t)

    t0 = time.time()
    results = []
    doc_token_cache.clear()

    tok_manifest = {}
    tok_path = Path('/content/drive/MyDrive/LRTIA/Results/corpus_expansion/tokenized_manifest_llama.jsonl')
    if tok_path.exists():
        with open(tok_path) as f:
            for line in f:
                d = json.loads(line)
                tok_manifest[d['document_id']] = d['file_path']

    for doc_id in tqdm(by_doc, desc=corpus_id):
        fp = tok_manifest.get(doc_id)
        if fp is None: continue

        try:
            full_ids = get_doc_tokens(fp)
        except Exception as e:
            print(f'  Error loading {doc_id}: {e}'); continue

        n_tok = len(full_ids)
        # Recompute target windows from target_position_fraction in Mistral token coords.
        rem_start = C
        rem_end = n_tok - TARGET_LEN
        if rem_end <= rem_start: continue

        for t in by_doc[doc_id]:
            frac = t.get('target_position_fraction', 0.5)
            ts = int(rem_start + frac * (rem_end - rem_start))
            te = ts + TARGET_LEN
            cs = ts - C
            ce = ts
            if cs < 0 or te > n_tok: continue

            ctx = full_ids[cs:ce]
            tgt = full_ids[ts:te]
            if len(ctx) < C or len(tgt) < 5: continue

            r = compute_corrected_curves(ctx, tgt)
            r['corpus_id'] = corpus_id
            r['document_id'] = doc_id
            r['target_id'] = t['target_id']
            r['target_frac'] = frac
            r['language'] = t['language']
            r['genre'] = t['genre']
            r['modality'] = t['modality']
            results.append(r)

    elapsed = time.time() - t0
    with open(cache_path, 'w') as f:
        json.dump(results, f)

    print(f'  {len(results)} results in {elapsed/60:.1f} min')

    if results:
        mean_delta = np.mean([np.mean(r['delta_ppl']) for r in results])
        curve = np.mean([r['delta_ppl'] for r in results], axis=0)
        alpha, r_val = fit_pl(np.array(curve))
        a_str = f'{alpha:.3f} (r={r_val:.3f})' if alpha else '—'
        print(f'  Mean Δ: {mean_delta:.4f}, α: {a_str}')

print('\nPhase complete!')

In [ ]:
# === Summary table for completed corpora ===
import matplotlib.pyplot as plt

print(f'\n{"="*90}')
print('CORPUS EXPANSION — Corrected Marginals Summary')
print(f'{"="*90}')
print(f'{"Corpus":<30} {"Genre":<20} {"Lang":<5} {"N":>5} {"TotalΔ":>8} {"α":>8} {"r":>8}')
print('-' * 88)

all_results = {}
for corpus_id in sorted(RUN_CORPORA):
    cp = BASE / f'{corpus_id}.json'
    if not cp.exists(): continue
    with open(cp) as f: results = json.load(f)
    if not results: continue
    all_results[corpus_id] = results
    
    curve = np.mean([r['delta_ppl'] for r in results], axis=0)
    total = np.mean(curve)
    alpha, r_val = fit_pl(np.array(curve))
    genre = results[0].get('genre', '?')
    lang = results[0].get('language', '?')
    a_str = f'{alpha:.3f}' if alpha else '—'
    r_str = f'{r_val:.3f}' if r_val else '—'
    print(f'{corpus_id:<30} {genre:<20} {lang:<5} {len(results):>5} {total:>8.4f} {a_str:>8} {r_str:>8}')

# Original Wiki range for comparison
print(f'\nOriginal Wiki range (Llama): α ≈ -0.73 to -0.88')

In [ ]:
# === Plots ===

if all_results:
    n = len(all_results)
    ncols = min(4, n)
    nrows = (n + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(6*ncols, 5*nrows), squeeze=False)
    axes_flat = axes.flatten()
    
    for idx, (corpus_id, results) in enumerate(sorted(all_results.items())):
        ax = axes_flat[idx]
        curve = np.mean([r['delta_ppl'] for r in results], axis=0)
        smooth = uniform_filter1d(curve, 5)
        ax.plot(range(1, len(smooth)+1), smooth, linewidth=2)
        ax.axhline(0, color='gray', linestyle=':', alpha=0.3)
        alpha, r_val = fit_pl(np.array(curve))
        title = corpus_id.replace('_', ' ')
        if alpha:
            title += f'\nα={alpha:.2f} (r={r_val:.2f})'
        ax.set_title(title, fontweight='bold', fontsize=10)
        ax.set_xlabel('Distance d')
        ax.set_ylabel('Corrected Δ')
        ax.grid(True, alpha=0.15)
    
    for idx in range(n, len(axes_flat)):
        axes_flat[idx].set_visible(False)
    
    plt.suptitle(f'Corpus Expansion Phase {PHASE}: Corrected Marginal Curves',
                 fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig(BASE / f'fig_expansion_phase{PHASE}.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Figure saved')